# Ψ₀ có "nhìn" cái lon không? Attention và che lon trên các frame eval vòng kín (`can_nav_pnp`)

Notebook lấy đúng các frame mà mô hình đã thấy trong lần đánh giá vòng kín (cell 6c của `colab_psi0_server.ipynb`), quanh lúc tay
phải chạm lon, và hỏi lại checkpoint Ψ₀ theo đúng cách server làm (cùng `Server`, cùng biến đổi ảnh, chuẩn hoá state, 10 bước
khử nhiễu, không RTC). Với mỗi frame có hai phép đo:

1. **Attention của action header.** Mỗi block `VLATransformerBlock` nối token action với token quan sát (token ảnh và chữ của
   Qwen3-VL) rồi tính attention chung. Notebook ghi lại trọng số của các hàng action trên các token ảnh (mỗi token là một ô 32×32
   pixel), lấy trung bình theo head, theo 30 bước action và theo các bước khử nhiễu, rồi phủ lên ảnh ego.
   - `can_lift` = (phần attention ảnh rơi vào các ô có lon) ÷ (phần diện tích các ô đó). Bằng 1: không chú ý lon hơn chỗ khác;
     lớn hơn 1: chú ý lon nhiều hơn.
2. **Che lon.** Tô màu sàn đè lên lon rồi dự đoán lại với cùng các seed nhiễu. `occ_ratio` = (action đổi bao nhiêu khi che lon)
   ÷ (action đổi bao nhiêu khi chỉ đổi seed nhiễu), đo trên 15 bước token SONIC được thực thi.
   - Khoảng 1: che lon không làm action đổi hơn nhiễu, tức mô hình gần như không dùng lon ở frame đó.
   - Lớn hơn nhiều so với 1: mô hình dùng hình ảnh lon để quyết định chuyển động.
   - `closure_orig → closure_occ`: lệnh nắm tay phải (0 mở, 1 nắm; gắp kích hoạt từ 0,3) trước và sau khi che.

**Yêu cầu:**
- Checkpoint trên Drive như notebook server (`My Drive/<DRIVE_RUNS>/finetune/...`).
- Thư mục bằng chứng eval trên Drive: `My Drive/<DRIVE_RUNS>/eval/<EVAL_NAME>/seed_*/` có `trace.npz` và `ego.mp4` (cell 6c tải lên).
- GPU **L4, A100, H100 hoặc G4**. Không cần server hay mô phỏng; có thể chạy trên runtime riêng.

| Cell | Việc |
|---|---|
| 1 | Cấu hình: checkpoint, lần eval, các seed, cửa sổ thời gian quanh lúc chạm lon |
| 2 | GPU, kết nối Drive, chọn checkpoint (giống notebook server) |
| 3 | Cài môi trường (giống notebook server), 10–15 phút |
| 4 | Ghi `.env` |
| 5 | Tải thư mục run và checkpoint từ Drive |
| 6 | Tải `trace.npz` + `ego.mp4` của các cảnh, chọn các frame có request quanh lúc chạm lon |
| 7 | Ghi script phân tích `probe.py` |
| 8 | Chạy phân tích: nạp mô hình 2–3 phút, sau đó vài giây mỗi frame |
| 9 | Bảng kết quả và hình heatmap từng cảnh |
| 10 | Tải kết quả lên `My Drive/<DRIVE_RUNS>/attn/<EVAL_NAME>/` |


In [ ]:
# @title 1. Cấu hình
# Only declares the settings: cell 2 exports them for the other cells, so after editing this cell run cell 2 as well.
CFG = {
    "PSI0_REPO": "https://github.com/physical-superintelligence-lab/Psi0.git",
    "PSI0_COMMIT": "2830f9367fc12ca49d9b4df4348726daae9db15d",   # the release of the SONIC 1.1 checkpoint, as in training
    "DRIVE_RUNS": "psi0/runs",  # on the Google Drive of the account running this notebook, relative to My Drive
    "EXP": "cannav160-s11",      # the EXP of colab_finetune_psi0_sonic11.ipynb
    "RUN_TS": "",               # empty: the run id in <DRIVE_RUNS>/<EXP>.run_ts (the finetune notebook's run)
    "CKPT_STEP": "",            # empty: the newest checkpoint of that run on Drive (must be the one that was evaluated)
    "EVAL_NAME": "",            # folder under <DRIVE_RUNS>/eval/; empty: <EXP>_ckpt<CKPT_STEP>_center_colab (cell 6c)
    "SEEDS": "",                # comma-separated seeds, e.g. "1000001,1000005"; empty: every SUCCESS / FAIL scene
    "PRE_S": "2.0",             # frames from this many seconds before the first hand-can contact ...
    "POST_S": "1.5",            # ... to this many after it, stopping at the grasp or when the can tilts past 10 deg
    "NUM_STEPS": "10",          # denoising steps, as serve_psi0_sonic_http (ServerConfig default)
}


In [ ]:
# @title 2. GPU, Google Drive, checkpoint
import io
import os
import pathlib
import re
import subprocess

from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload

os.environ["NB_READY"] = ""  # set again at the end, once everything below is exported
os.environ.update(CFG)
os.environ["PSI_HOME"] = "/content/psi_home"

name, cc = [s.strip() for s in subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,compute_cap", "--format=csv,noheader"], text=True).splitlines()[0].split(",")]
print(f"GPU: {name}, compute capability {cc}")
if float(cc) < 8.0:
    raise RuntimeError("Need an Ampere or newer GPU (L4 / A100 / H100 / G4): the model runs in bf16.")


class Drive:
    """My Drive of the account running this notebook, through the Drive API.

    Not drive.mount: a file deleted through the mount goes to the Drive trash and keeps counting
    against the quota (10 GB per rotated checkpoint), and the mount uploads in the background, so a
    checkpoint could look complete on Drive before all of it is there. Here upload() returns once
    the bytes are on Drive, and delete() is permanent.
    """
    FOLDER = "application/vnd.google-apps.folder"

    def __init__(self):
        auth.authenticate_user()
        self.svc = build("drive", "v3", cache_discovery=False)

    def children(self, folder_id):
        out, token = [], None
        while True:
            r = self.svc.files().list(q=f"'{folder_id}' in parents and trashed = false", pageSize=1000,
                                      pageToken=token, fields="nextPageToken, files(id, name, mimeType, size)"
                                      ).execute(num_retries=5)
            out += r["files"]
            token = r.get("nextPageToken")
            if not token:
                return out

    def child(self, folder_id, name):
        return next((f for f in self.children(folder_id) if f["name"] == name), None)

    def mkdir(self, parent, name):
        return self.svc.files().create(body={"name": name, "mimeType": self.FOLDER, "parents": [parent]},
                                       fields="id").execute(num_retries=5)["id"]

    def path(self, rel, create=False):
        """Id of the file or folder at `rel` (relative to My Drive), or None."""
        fid = "root"
        for part in pathlib.PurePosixPath(rel).parts:
            f = self.child(fid, part)
            if f is None and not create:
                return None
            fid = f["id"] if f else self.mkdir(fid, part)
        return fid

    def upload(self, local, parent, name=None):
        """Upload a file, or a directory recursively, into `parent`; replaces a file of the same name."""
        local = pathlib.Path(local)
        name = name or local.name
        if local.is_dir():
            fid = self.mkdir(parent, name)
            for p in sorted(local.iterdir()):
                if not p.is_symlink():
                    self.upload(p, fid)
            return fid
        size = local.stat().st_size
        old = self.child(parent, name)
        if size == 0:  # nothing to send; an empty upload body is rejected
            return old["id"] if old else self.svc.files().create(
                body={"name": name, "parents": [parent]}, fields="id").execute(num_retries=5)["id"]
        media = MediaFileUpload(str(local), resumable=size > 8 * 2**20, chunksize=256 * 2**20)
        if old:
            req = self.svc.files().update(fileId=old["id"], media_body=media, fields="id, size")
        else:
            req = self.svc.files().create(body={"name": name, "parents": [parent]}, media_body=media,
                                          fields="id, size")
        if media.resumable():
            resp = None
            while resp is None:
                _, resp = req.next_chunk(num_retries=5)
        else:
            resp = req.execute(num_retries=5)
        if int(resp.get("size", -1)) != size:
            raise IOError(f"{local}: Drive holds {resp.get('size')} bytes, expected {size}")
        return resp["id"]

    def download(self, file_id, local):
        local = pathlib.Path(local)
        local.parent.mkdir(parents=True, exist_ok=True)
        with open(local, "wb") as fh:
            dl = MediaIoBaseDownload(fh, self.svc.files().get_media(fileId=file_id), chunksize=64 * 2**20)
            done = False
            while not done:
                _, done = dl.next_chunk(num_retries=5)

    def download_tree(self, item, local):
        """`item`: an entry of children(); a folder is downloaded recursively."""
        if item["mimeType"] == self.FOLDER:
            for c in self.children(item["id"]):
                self.download_tree(c, pathlib.Path(local) / c["name"])
        else:
            self.download(item["id"], local)

    def read_text(self, rel):
        fid = self.path(rel)
        if fid is None:
            return ""
        buf = io.BytesIO()
        dl = MediaIoBaseDownload(buf, self.svc.files().get_media(fileId=fid))
        done = False
        while not done:
            _, done = dl.next_chunk(num_retries=5)
        return buf.getvalue().decode()

    def write_text(self, rel, text):
        rel = pathlib.PurePosixPath(rel)
        tmp = pathlib.Path("/tmp") / rel.name
        tmp.write_text(text)
        self.upload(tmp, self.path(str(rel.parent), create=True))

    def rename(self, file_id, name):
        self.svc.files().update(fileId=file_id, body={"name": name}).execute(num_retries=5)

    def delete(self, file_id):
        self.svc.files().delete(fileId=file_id).execute(num_retries=5)  # permanent: skips the trash

    def free_bytes(self):
        q = self.svc.about().get(fields="storageQuota").execute(num_retries=5)["storageQuota"]
        return int(q["limit"]) - int(q["usage"]) if "limit" in q else None



gd = Drive()
run_ts = CFG["RUN_TS"] or gd.read_text(f"{CFG['DRIVE_RUNS']}/{CFG['EXP']}.run_ts").strip()
assert run_ts, f"no run id: My Drive/{CFG['DRIVE_RUNS']}/{CFG['EXP']}.run_ts not found - set RUN_TS"
finetune = gd.path(f"{CFG['DRIVE_RUNS']}/finetune")
run = next((f for f in (gd.children(finetune) if finetune else [])
            if f["mimeType"] == gd.FOLDER and f["name"].endswith(f".{run_ts}")), None)
assert run, f"no run folder ending in .{run_ts} under My Drive/{CFG['DRIVE_RUNS']}/finetune"
ck = gd.child(run["id"], "checkpoints")
on_drive = {int(m.group(1)): f for f in (gd.children(ck["id"]) if ck else [])
            if (m := re.fullmatch(r"ckpt_(\d+)", f["name"]))}
assert on_drive, f"{run['name']} has no complete checkpoint on Drive yet"
step = int(CFG["CKPT_STEP"]) if CFG["CKPT_STEP"] else max(on_drive)
assert step in on_drive, f"ckpt_{step} is not on Drive; there: {sorted(on_drive)}"
os.environ.update(RUN_DIR=f"/content/runs/finetune/{run['name']}", CKPT_STEP=str(step))
print(f"run {run['name']}\ncheckpoint ckpt_{step} (on Drive: {sorted(on_drive)})")
os.environ["NB_READY"] = "1"

In [ ]:
%%bash
# 3. Environment: /content/Psi0 at PSI0_COMMIT + .venv-psi (Python 3.11, torch 2.7, flash-attn or SDPA)
set -euo pipefail
: "${NB_READY:?run cell 2 first (after a runtime restart or an edit of cell 1, re-run cells 1 and 2)}"
# Colab's own values would leak into the venv; UV_SYSTEM_PYTHON would point uv at Colab's python
unset PYTHONPATH MPLBACKEND UV_SYSTEM_PYTHON
export GIT_LFS_SKIP_SMUDGE=1 PATH="$HOME/.local/bin:$PATH"
# uv.lock at PSI0_COMMIT does not parse: its relock (6b4a5cc) left a duplicated table and packages
# listed under two registries. This is the last lock that parses, all on PyPI; the psi group is the
# same since, and the only newer dependencies (decord2, wandb[media]) are not used by this training.
GOOD_LOCK_COMMIT=47993ee31926235b2a640cc554a8c93f66220a3a
# the wheel scripts/train/Dockerfile uses: cu12 / torch 2.7 / cxx11abi TRUE / cp311, sm80 + sm90 kernels
FLASH_ATTN_WHEEL="https://github.com/Dao-AILab/flash-attention/releases/download/v2.7.4.post1/flash_attn-2.7.4.post1+cu12torch2.7cxx11abiTRUE-cp311-cp311-linux_x86_64.whl"
# Blackwell (G4 = RTX PRO 6000, compute capability 12.0): the locked torch 2.7.0 is the CUDA 12.6 build,
# which has no sm_100/sm_120 kernels, and flash-attn 2 publishes no Blackwell wheel. There the same
# torch 2.7.0 comes from the CUDA 12.8 index (README troubleshooting 5) and the VLM uses PyTorch's
# SDPA attention: the same attention, computed by a different kernel.
CC=$(nvidia-smi --query-gpu=compute_cap --format=csv,noheader | head -n 1)
if [ "${CC%%.*}" -ge 10 ]; then VARIANT=cu128-sdpa; else VARIANT=cu126-flash; fi
echo "GPU compute capability $CC -> $VARIANT"

mkdir -p /content/Psi0 && cd /content/Psi0
[ -d .git ] || { git init -q && git remote add origin "$PSI0_REPO"; }
if [ "$(git rev-parse -q --verify HEAD || true)" != "$PSI0_COMMIT" ]; then
    git fetch -q --depth 1 origin "$PSI0_COMMIT"
    git checkout -q --force FETCH_HEAD
fi
echo "Psi0 @ $(git rev-parse --short HEAD)"

command -v uv >/dev/null || curl -LsSf https://astral.sh/uv/install.sh | sh
if [ -f .venv-psi/.synced ]; then
    echo "locked dependencies already installed"
else
    uv python install 3.11
    [ -d .venv-psi ] || uv venv .venv-psi --python 3.11
    source .venv-psi/bin/activate
    git fetch -q --depth 1 origin "$GOOD_LOCK_COMMIT"
    git show FETCH_HEAD:uv.lock > uv.lock
    # --frozen: install the lock as-is; --inexact: keep what is installed on top of it (flash-attn,
    # the CUDA 12.8 torch)
    uv sync --active --frozen --group psi --inexact
    touch .venv-psi/.synced
fi
# Checked against what is installed rather than a marker, so a half-finished run repairs itself.
if [ "$VARIANT" = cu128-sdpa ]; then
    if [ "$(.venv-psi/bin/python -c 'import torch; print(torch.version.cuda)')" != 12.8 ]; then
        # the +cu128 local version is required: a plain ==2.7.0 is already satisfied by the
        # locked 2.7.0+cu126, and uv would keep that build
        uv pip install --python .venv-psi/bin/python torch==2.7.0+cu128 torchvision==0.22.0+cu128 \
            --index-url https://download.pytorch.org/whl/cu128
    fi
elif ! .venv-psi/bin/python -c "import flash_attn" 2>/dev/null; then
    uv pip install --python .venv-psi/bin/python "$FLASH_ATTN_WHEEL"
fi
# the HTTP server needs the serve group (fastapi, uvicorn), pinned to the lock's versions. Installed
# by itself: a uv sync would put back the locked CUDA 12.6 torch over the 12.8 one.
if ! .venv-psi/bin/python -c "import fastapi, uvicorn" 2>/dev/null; then
    uv pip install --python .venv-psi/bin/python fastapi==0.119.1 uvicorn==0.38.0
fi
if [ "$VARIANT" = cu128-sdpa ]; then
    # finetune.py hard-codes flash_attention_2 for the VLM (the serving code already falls back to sdpa)
    sed -i 's/attn_implementation="flash_attention_2"/attn_implementation="sdpa"/' src/psi/trainers/finetune.py
fi
grep -n 'attn_implementation=' src/psi/trainers/finetune.py

# serve_psi0_simple at this commit passes no pooled_projections to combined_temb (SONIC 1.1) checkpoints
.venv-psi/bin/python - src/psi/deploy/serve_psi0_simple.py <<'PATCH'
# Patch for Psi0 2830f93: serve_psi0_simple (behind serve_psi0_sonic_http) passes no pooled_projections, but the
# SONIC 1.1 checkpoints are combined_temb models (--model.combined-temb, CLIP pooled instruction embedding), so
# /act fails with "linear(): argument 'input' must be Tensor, not NoneType". Same logic as serve_psi0_sonic.py's
# PooledTextEncoderCache: the instruction embeddings cached at training time, unseen ones from the frozen CLIP.
import sys
from pathlib import Path

p = Path(sys.argv[1] if len(sys.argv) > 1 else "/content/Psi0/src/psi/deploy/serve_psi0_simple.py")
s = p.read_text()
MARK = "# [can_nav patch] pooled_projections"
if MARK in s:
    print("serve_psi0_simple.py: already patched")
    sys.exit(0)
init_anchor = "        self.model.eval()\n"
call_anchor = "                    instructions=[instruction], # [Task] * B\n"
assert s.count(init_anchor) == 1 and s.count(call_anchor) == 1, "serve_psi0_simple.py changed: patch anchors not found"
s = s.replace(init_anchor, init_anchor + f'''
        {MARK}: combined_temb models need a pooled instruction embedding (as serve_psi0_sonic.py)
        self._pooled_cache, self._pooled_model, self._pooled_tok = {{}}, None, None
        self._pooled_cfg = launch_config.model if getattr(launch_config.model, "pooled_text_encoder", None) else None
        if self._pooled_cfg is not None:
            _rel = getattr(self._pooled_cfg, "pooled_cache_path", None)
            _file = None if _rel is None else (_rel if osp.isabs(_rel) else str(run_dir / _rel))
            if _file is not None and osp.exists(_file):
                self._pooled_cache = {{k: v.to(self.device) for k, v in torch.load(_file, map_location="cpu").items()}}
                overwatch.info(f"Loaded pooled text cache ({{len(self._pooled_cache)}} entries) <- {{_file}}")
            else:
                overwatch.info(f"No pooled text cache at {{_file}}: instructions are encoded with the frozen text encoder")
''', 1)
s = s.replace(call_anchor, call_anchor + "                    pooled_projections=self._pooled(instruction),  " + MARK + "\n", 1)
method = f'''
    {MARK}
    @torch.no_grad()
    def _pooled(self, instruction: str):
        if self._pooled_cfg is None:
            return None
        key = instruction.lower()
        emb = self._pooled_cache.get(key)
        if emb is None:
            if self._pooled_model is None:
                assert self._pooled_cfg.pooled_text_encoder == "clip", self._pooled_cfg.pooled_text_encoder
                from transformers import CLIPTextModelWithProjection, CLIPTokenizer
                path = self._pooled_cfg.pooled_text_encoder_path
                self._pooled_tok = CLIPTokenizer.from_pretrained(path)
                self._pooled_model = CLIPTextModelWithProjection.from_pretrained(path, dtype=torch.bfloat16).to(self.device).eval()
                assert self._pooled_model.config.projection_dim == self._pooled_cfg.pooled_projection_dim
            toks = self._pooled_tok([key], padding=True, truncation=True, max_length=self._pooled_tok.model_max_length,
                                    return_tensors="pt").to(self.device)
            emb = self._pooled_model(**toks).text_embeds[0]
            self._pooled_cache[key] = emb
            overwatch.info(f"pooled cache MISS -> encoded instruction: {{instruction!r}}")
        return emb.unsqueeze(0)  # (1, D)

    def _order_images('''
anchor3 = "\n    def _order_images("
assert s.count(anchor3) == 1
s = s.replace(anchor3, method, 1)
p.write_text(s)
print("serve_psi0_simple.py: patched (pooled_projections for combined_temb checkpoints)")
PATCH
source .venv-psi/bin/activate
python - "$VARIANT" <<'PY'
import sys, torch, psi, fastapi, uvicorn
from psi.data.lerobot.compat import LEROBOT_LAYOUT
# fails here, not mid-training, when the build has no kernel for this GPU
x = torch.randn(256, 256, device="cuda", dtype=torch.bfloat16)
(x @ x).sum().item()
if sys.argv[1] == "cu126-flash":
    import flash_attn
    from flash_attn import flash_attn_func
    q = torch.randn(1, 64, 4, 64, device="cuda", dtype=torch.bfloat16)
    flash_attn_func(q, q, q)
    attn = f"flash_attn {flash_attn.__version__}"
else:
    q = torch.randn(1, 4, 64, 64, device="cuda", dtype=torch.bfloat16)
    torch.nn.functional.scaled_dot_product_attention(q, q, q).sum().item()
    attn = "sdpa"
print(f"torch {torch.__version__} (cuda {torch.version.cuda}) | {attn} | lerobot layout {LEROBOT_LAYOUT} | "
      f"{torch.cuda.get_device_name(0)}")
PY

In [ ]:
%%bash
# 4. .env: scripts/train.py asserts load_dotenv() finds one. Secrets stay in the process
# environment (cell 2) and are never written to disk.
set -euo pipefail
: "${NB_READY:?run cell 2 first (after a runtime restart or an edit of cell 1, re-run cells 1 and 2)}"
mkdir -p "$PSI_HOME"/data "$PSI_HOME"/cache/hf "$PSI_HOME"/cache/checkpoints
cat > /content/Psi0/.env <<EOF
PSI_HOME=$PSI_HOME
DATA_HOME=$PSI_HOME/data
HF_HOME=$PSI_HOME/cache/hf
TORCH_HOME=$PSI_HOME/cache/hf
HF_LEROBOT_HOME=$PSI_HOME/data/lerobot
OMP_NUM_THREADS=8
TOKENIZERS_PARALLELISM=false
CUDA_LAUNCH_BLOCKING=0
TF_CPP_MIN_LOG_LEVEL=3
AV_LOG_LEVEL=quiet
PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION=python
NO_ALBUMENTATIONS_UPDATE=1
EOF
cat /content/Psi0/.env

In [ ]:
# @title 5. Tải thư mục run và checkpoint từ Drive
import os
import pathlib

assert os.environ.get("NB_READY"), "run cell 2 first (after a runtime restart or an edit of cell 1, re-run cells 1 and 2)"
run_dir = pathlib.Path(os.environ["RUN_DIR"])


def fetch(item, local):
    """Download a Drive file or folder unless an identical-size local copy exists."""
    local = pathlib.Path(local)
    if item["mimeType"] == gd.FOLDER:
        for c in gd.children(item["id"]):
            fetch(c, local / c["name"])
    elif not (local.exists() and local.stat().st_size == int(item.get("size", -1))):
        print(f"  {local.relative_to(run_dir.parent)} ({int(item.get('size', 0)) / 2**30:.2f} GB)")
        gd.download(item["id"], local)


for f in gd.children(run["id"]):
    if f["mimeType"] != gd.FOLDER:
        fetch(f, run_dir / f["name"])
fetch(on_drive[step], run_dir / "checkpoints" / f"ckpt_{step}")
print("ready:", run_dir, "| ckpt", step)

In [ ]:
# @title 6. Tải bằng chứng eval và chọn các frame quanh lúc tay chạm lon
# The frames are the ones the policy was asked about: rows of trace.npz with new_chunk set, the ego.mp4 frame of that
# row (what the policy saw, 320x240), and the state closed_loop_eval.py sent (qpos[7:50] in the 0810 joint order).
import json
import os
import pathlib
import re
import subprocess
import sys

import numpy as np

assert os.environ.get("NB_READY"), "run cell 2 first (after a runtime restart or an edit of cell 1, re-run cells 1 and 2)"
try:
    import imageio.v2 as iio
    import imageio_ffmpeg  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "imageio", "imageio-ffmpeg"], check=True)
    import imageio.v2 as iio

EVAL_NAME = CFG["EVAL_NAME"] or f"{CFG['EXP']}_ckpt{os.environ['CKPT_STEP']}_center_colab"
PRE_S, POST_S = float(CFG["PRE_S"]), float(CFG["POST_S"])
# export_psi0.state_permutation(): MJCF [lleg rleg waist larm lhand rarm rhand] -> 0810 [lleg rleg waist larm rarm lhand rhand]
PERM = list(range(22)) + list(range(29, 36)) + list(range(22, 29)) + list(range(36, 43))
WORK = pathlib.Path("/content/attn")
WORK.mkdir(parents=True, exist_ok=True)   # cell 7 writes probe.py here
EV = WORK / "eval" / EVAL_NAME


def select_frames(trace):
    """Request rows from PRE_S before the first hand-can contact to POST_S after it, cut at the grasp or a 10 deg tilt."""
    t, c, bp = trace["t"], trace["contacts"], trace["box_pose"]
    if not c[:, 1].any():
        return None, []
    i0 = int(np.argmax(c[:, 1]))
    up_z = 1 - 2 * (bp[:, 4] ** 2 + bp[:, 5] ** 2)          # can z axis . world z (box_pose = xyz + quat wxyz)
    event = np.where((up_z < np.cos(np.radians(10))) | c[:, 0])[0]
    event = event[event >= i0]
    t_end = min(t[i0] + POST_S, t[event[0]] if len(event) else np.inf)
    return i0, [int(k) for k in np.where(trace["new_chunk"])[0] if t[i0] - PRE_S <= t[k] <= t_end]


ev_id = gd.path(f"{CFG['DRIVE_RUNS']}/eval/{EVAL_NAME}")
assert ev_id, f"không thấy My Drive/{CFG['DRIVE_RUNS']}/eval/{EVAL_NAME}: chạy cell 6c của notebook server, hoặc đặt EVAL_NAME"
scenes = sorted((f for f in gd.children(ev_id)
                 if f["mimeType"] == gd.FOLDER and re.fullmatch(r"seed_\d+_(SUCCESS|FAIL_.+)", f["name"])),
                key=lambda f: f["name"])
if CFG["SEEDS"]:
    keep = {s.strip() for s in CFG["SEEDS"].split(",")}
    scenes = [f for f in scenes if f["name"].split("_")[1] in keep]
assert scenes, f"không có cảnh SUCCESS / FAIL nào trong My Drive/{CFG['DRIVE_RUNS']}/eval/{EVAL_NAME}"

images, states, meta = [], [], []
for f in scenes:
    local = EV / f["name"]
    for name in ("trace.npz", "ego.mp4"):
        item = gd.child(f["id"], name)
        if item and not (local / name).exists():
            gd.download(item["id"], local / name)
    if not ((local / "trace.npz").exists() and (local / "ego.mp4").exists()):
        print(f"{f['name']}: thiếu trace.npz hoặc ego.mp4, bỏ qua")
        continue
    trace = np.load(local / "trace.npz")
    i0, want = select_frames(trace)
    if not want:
        print(f"{f['name']}: tay không chạm lon, bỏ qua")
        continue
    frames = {}
    with iio.get_reader(str(local / "ego.mp4"), "ffmpeg") as reader:   # ego.mp4 frame k == trace row k
        for k, img in enumerate(reader):
            if k in want:
                frames[k] = img
            if k >= want[-1]:
                break
    t = trace["t"]
    for k in want:
        images.append(frames[k])
        states.append(trace["qpos"][k, 7:50][PERM])
        meta.append(dict(scene=f["name"], seed=int(f["name"].split("_")[1]), success=f["name"].endswith("SUCCESS"),
                         frame=k, t=round(float(t[k]), 2), t_rel=round(float(t[k] - t[i0]), 2)))
    print(f"{f['name']}: chạm lon lúc {t[i0]:.1f} s, {len(want)} frame "
          f"({', '.join(f'{t[k] - t[i0]:+.1f}' for k in want)} s)")

np.savez_compressed(WORK / "frames.npz", images=np.stack(images), states=np.stack(states).astype(np.float32))
(WORK / "frames.json").write_text(json.dumps(meta, indent=1))
print(f"{len(meta)} frame từ {len({m['scene'] for m in meta})} cảnh -> {WORK}/frames.npz")


In [ ]:
%%writefile /content/attn/probe.py
"""Psi0 attention + can-occlusion probe on closed-loop eval frames (colab_psi0_attention.ipynb, cell 7).

  probe.py WORK_DIR      reads WORK_DIR/frames.npz + frames.json (cell 6), writes WORK_DIR/out/

Run from /content/Psi0 (its .env) with RUN_DIR and CKPT_STEP in the environment. Each frame is
predicted exactly as serve_psi0_sonic_http does it (same Server object, transforms, normalizers,
pooled instruction embedding, open loop, NUM_STEPS denoising steps), with two additions:
  1. attention: JointVLAAttnProcessor's weights of the action rows over the observation tokens,
     per action block, averaged over heads, action rows and denoising steps;
  2. occlusion: the red can painted over with the surrounding floor colour, then the same
     prediction under the same noise seeds, compared with the change two noise seeds alone make.
"""
import json
import os
import sys
import time
from itertools import combinations
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()

import numpy as np  # noqa: E402
import torch  # noqa: E402
from PIL import Image, ImageFilter  # noqa: E402
from torchvision.transforms import v2  # noqa: E402

import psi.models.psi0 as P  # noqa: E402
from psi.deploy.serve_psi0_simple import Server  # noqa: E402

WORK = Path(sys.argv[1])
OUT = WORK / "out"
OUT.mkdir(parents=True, exist_ok=True)

# episode.INSTRUCTION of can_nav_pnp, lower-cased as the server does before tokenizing
INSTRUCTION = ("walk around the box in front of you to the low stand, squat down, pick up the can from the stand with "
               "your right hand, stand up, walk to the table, put the can in the tray on the table, then return to "
               "the home pose.")
NOISE_SEEDS = (0, 1, 2)
EXEC = 15                                # rows the client executes per chunk (closed_loop_eval --exec-steps)
TOKEN, RHAND = slice(0, 64), slice(71, 78)
RIGHT_HAND_CLOSED = np.array([0.0, 0.0, -1.75, 1.57, 1.75, 1.57, 1.75])  # sonic_policy.py
CAN_CELL_FRAC = 0.01                     # a token cell is a "can cell" if >= 1 % of its pixels are can


def log(msg):
    print(f"[probe] {msg}", flush=True)


def red_mask(img) -> np.ndarray:
    """The can's red body (the floor is blue, the table brown, the box yellow, the robot white)."""
    a = np.asarray(img).astype(np.int16)
    return (a[..., 0] > 45) & (a[..., 0] > 2 * a[..., 1]) & (a[..., 0] > 2 * a[..., 2])


def occlude(img: np.ndarray):
    """The can (red body, dilated over its white top and edges) filled with the median colour around it."""
    m = red_mask(img)
    if m.sum() < 3:
        return None
    mask = Image.fromarray(m.astype(np.uint8) * 255)
    inner = np.asarray(mask.filter(ImageFilter.MaxFilter(9))) > 0
    ring = (np.asarray(mask.filter(ImageFilter.MaxFilter(25))) > 0) & ~inner
    out = img.copy()
    out[inner] = np.median(img[ring], axis=0).astype(np.uint8)
    return out


# -- attention capture ---------------------------------------------------------------------------------------------
# JointVLAAttnProcessor concatenates [action tokens | observation tokens] and calls psi0.py's
# F.scaled_dot_product_attention, which returns no weights. The wrapper below notes where the
# action tokens end; the F stand-in runs the same SDPA and, while recording, also computes the
# softmax weights of the action rows. Everything else in psi0.py gets the real torch.nn.functional.
REC = {"on": False, "s1": None, "maps": []}
_orig_call = P.JointVLAAttnProcessor.__call__


def _call(self, attn, hidden_states, encoder_hidden_states=None, attention_mask=None, *args, **kwargs):
    REC["s1"] = hidden_states.shape[1]
    return _orig_call(self, attn, hidden_states, encoder_hidden_states, attention_mask, *args, **kwargs)


class _F:
    def __getattr__(self, name):
        return getattr(torch.nn.functional, name)

    @staticmethod
    def scaled_dot_product_attention(q, k, v, attn_mask=None, dropout_p=0.0, is_causal=False, **kw):
        out = torch.nn.functional.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask, dropout_p=dropout_p,
                                                               is_causal=is_causal, **kw)
        s1, REC["s1"] = REC["s1"], None
        if REC["on"] and s1 is not None:
            w = torch.matmul(q[:, :, :s1].float(), k.float().transpose(-1, -2)) * q.shape[-1] ** -0.5
            if attn_mask is not None:
                w = w.masked_fill(~attn_mask.bool(), float("-inf"))
            REC["maps"].append(w.softmax(-1).mean(1)[0, :, s1:].cpu())  # (action rows, obs tokens)
        return out


P.JointVLAAttnProcessor.__call__ = _call
P.F = _F()

# -- the served model ----------------------------------------------------------------------------------------------
t0 = time.time()
server = Server("psi0", Path(os.environ["RUN_DIR"]), os.environ["CKPT_STEP"], "cuda:0", False, None,
                num_inference_steps=int(os.environ.get("NUM_STEPS") or 10))
model = server.model
n_blocks = len(model.action_header.transformer_blocks)
img_token_id = getattr(model.vlm_model.config, "image_token_id", None) \
    or model.vlm_processor.tokenizer.convert_tokens_to_ids("<|image_pad|>")
VLM = {}


def _vlm_pre(module, args, kwargs):
    VLM["ids"], VLM["thw"] = kwargs["input_ids"][0].cpu().numpy(), kwargs["image_grid_thw"][0].cpu().numpy()


model.vlm_model.register_forward_pre_hook(_vlm_pre, with_kwargs=True)
tf = v2.Compose([server.model_transform.resize(), server.model_transform.center_crop()])
pooled = server._pooled(INSTRUCTION) if hasattr(server, "_pooled") else None
log(f"model ready in {time.time() - t0:.0f} s: {n_blocks} action blocks, {server.num_inference_steps} denoising "
    f"steps, chunk {server.Tp}x{server.Da}, pooled instruction {'yes' if pooled is not None else 'no'}")


@torch.inference_mode()
def predict(image: Image.Image, state: np.ndarray, seed: int, record=False):
    REC["on"], REC["maps"], REC["s1"] = record, [], None
    torch.manual_seed(seed)
    raw = model.predict_action(observations=[[image]], states=server._prepare_states({"states": state}).unsqueeze(0),
                               instructions=[INSTRUCTION], num_inference_steps=server.num_inference_steps,
                               traj2ds=None, pooled_projections=pooled)
    REC["on"] = False
    raw = raw.reshape(-1, server.Da).float().cpu().numpy()
    return raw, server.maxmin.denormalize(raw)


def closure(act) -> float:
    """Commanded right-hand closure over the executed rows (0 open, 1 closed; the grasp assist needs >= 0.3)."""
    return float(np.mean(act[:EXEC, RHAND][:, 2:] / RIGHT_HAND_CLOSED[2:]))


def rms(a, b) -> float:
    return float(np.sqrt(np.mean((a[:EXEC, TOKEN] - b[:EXEC, TOKEN]) ** 2)))


def can_stats(cm: np.ndarray, can: np.ndarray) -> tuple[float, float]:
    """Share of the image attention on can cells, and that share over the can cells' area share."""
    cells = can >= CAN_CELL_FRAC
    share = float(cm[cells].sum() / cm.sum()) if cells.any() else float("nan")
    return share, share / cells.mean() if cells.any() else float("nan")


data = np.load(WORK / "frames.npz")
meta = json.loads((WORK / "frames.json").read_text())
rows, cellmaps, canfracs, shown, occluded = [], [], [], [], []
for i, m in enumerate(meta):
    t1 = time.time()
    img_np, state = data["images"][i], data["states"][i]
    image = tf(Image.fromarray(img_np))

    # 1. attention, recorded on the first noise seed
    raw0, act0 = predict(image, state, NOISE_SEEDS[0], record=True)
    maps = torch.stack(REC["maps"])
    assert maps.shape[0] % n_blocks == 0, f"{maps.shape[0]} attention calls for {n_blocks} blocks"
    maps = maps.view(maps.shape[0] // n_blocks, n_blocks, *maps.shape[1:])     # (steps, blocks, rows, obs)
    if maps.shape[2] == server.Tp + 1:                                          # state_as_action_token: row 0
        maps = maps[:, :, 1:]
    a = maps.mean(dim=(0, 2)).numpy()                                           # (blocks, obs tokens)
    ids, thw = VLM["ids"], VLM["thw"]
    n_vlm = len(ids)
    img_cols = np.where(ids == img_token_id)[0]
    gh, gw = int(thw[1]) // 2, int(thw[2]) // 2                                 # 2x2 patch merge -> 32 px cells
    assert len(img_cols) == gh * gw, f"{len(img_cols)} image tokens for a {gh}x{gw} grid"
    cm = a[:, img_cols].reshape(n_blocks, gh, gw)
    can = np.asarray(Image.fromarray(red_mask(image).astype(np.uint8) * 255).resize((gw, gh), Image.BOX)) / 255.0
    share_img = a[:, img_cols].sum(1)
    share_text = a[:, :n_vlm].sum(1) - share_img
    share_other = a[:, n_vlm:].sum(1)
    can_share0, can_lift0 = can_stats(cm[0], can)
    can_share_all, can_lift_all = can_stats(cm.mean(0), can)
    _, can_lift_last = can_stats(cm[-1], can)

    # 2. occlusion under the same noise seeds
    occ = occlude(img_np)
    res = dict(m, can_px=int(red_mask(img_np).sum()), can_cells=int((can >= CAN_CELL_FRAC).sum()),
               attn_img_b0=float(share_img[0]), attn_text_b0=float(share_text[0]), attn_other_b0=float(share_other[0]),
               attn_img_mean=float(share_img.mean()), can_share_b0=can_share0, can_lift_b0=can_lift0,
               can_share_mean=can_share_all, can_lift_mean=can_lift_all, can_lift_last=can_lift_last,
               closure=closure(act0))
    if occ is not None:
        image_x = tf(Image.fromarray(occ))
        base = [(raw0, act0)] + [predict(image, state, s) for s in NOISE_SEEDS[1:]]
        masked = [predict(image_x, state, s) for s in NOISE_SEEDS]
        d_occ = float(np.mean([rms(b[0], x[0]) for b, x in zip(base, masked)]))
        d_noise = float(np.mean([rms(base[p][0], base[q][0]) for p, q in combinations(range(len(base)), 2)]))
        res.update(d_occ=d_occ, d_noise=d_noise, occ_ratio=d_occ / d_noise if d_noise > 0 else float("nan"),
                   closure_orig=float(np.mean([closure(b[1]) for b in base])),
                   closure_occ=float(np.mean([closure(x[1]) for x in masked])))
    rows.append(res)
    cellmaps.append(cm.astype(np.float16))
    canfracs.append(can.astype(np.float16))
    shown.append(np.asarray(image))
    occluded.append(occ if occ is not None else np.zeros_like(img_np))
    log(f"{i + 1}/{len(meta)} {m['scene'][:24]} t{m['t_rel']:+.1f}s: can {res['can_px']} px, "
        f"attention on image {share_img[0]:.2f} (block 0), can lift b0 {can_lift0:.2f} / mean {can_lift_all:.2f}"
        + (f", occlusion x{res['occ_ratio']:.2f} of noise, closure {res['closure_orig']:.2f} -> {res['closure_occ']:.2f}"
           if occ is not None else ", can not visible: no occlusion test") + f" ({time.time() - t1:.1f} s)")

np.savez_compressed(OUT / "attn.npz", cellmaps=np.stack(cellmaps), canfrac=np.stack(canfracs),
                    images=np.stack(shown), occluded=np.stack(occluded))
(OUT / "metrics.json").write_text(json.dumps(rows, indent=1))
log(f"done: {len(rows)} frames -> {OUT}")


In [ ]:
# @title 8. Chạy phân tích (nạp mô hình 2-3 phút, sau đó vài giây mỗi frame)
import os
import pathlib
import subprocess

assert os.environ.get("NB_READY"), "run cell 2 first (after a runtime restart or an edit of cell 1, re-run cells 1 and 2)"
WORK = pathlib.Path("/content/attn")
assert (WORK / "frames.npz").exists(), "chạy cell 6 trước"
assert (WORK / "probe.py").exists(), "chạy cell 7 trước"
env = {k: v for k, v in os.environ.items() if k not in ("PYTHONPATH", "MPLBACKEND")}
PROBE_LOG = WORK / "probe.log"
with open(PROBE_LOG, "w") as log:
    p = subprocess.Popen(["/content/Psi0/.venv-psi/bin/python", "-u", str(WORK / "probe.py"), str(WORK)],
                         cwd="/content/Psi0", env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:   # the model code logs a lot: show only the probe's own lines here, all of it in probe.log
        log.write(line)
        if line.startswith("[probe]"):
            print(line[8:], end="", flush=True)
if p.wait() != 0:
    print(open(PROBE_LOG, errors="replace").read()[-5000:])
    raise RuntimeError(f"probe.py exited with code {p.returncode}, see the log above ({PROBE_LOG})")


In [ ]:
# @title 9. Kết quả: bảng và heatmap từng cảnh
import json
import pathlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

OUT = pathlib.Path("/content/attn/out")
rows = json.loads((OUT / "metrics.json").read_text())
arr = np.load(OUT / "attn.npz")
df = pd.DataFrame(rows)
df["kết quả"] = np.where(df["success"], "thành công", "thất bại")
for col in ("d_occ", "d_noise", "occ_ratio", "closure_orig", "closure_occ"):
    if col not in df:
        df[col] = np.nan

cols = ["kết quả", "can_px", "attn_img_b0", "can_lift_b0", "can_lift_mean", "can_lift_last", "occ_ratio",
        "closure_orig", "closure_occ"]
display(Markdown("### Trung bình theo kết quả của cảnh\n"
                 "`can_lift`: 1 = không chú ý lon hơn chỗ khác. `occ_ratio`: khoảng 1 = che lon không làm action đổi hơn nhiễu."))
display(df[cols].groupby("kết quả").mean(numeric_only=True).round(2))

display(Markdown("### Theo thời gian so với lúc chạm lon (âm = trước khi chạm)"))
df["pha"] = pd.cut(df["t_rel"], [-99, -1.0, 0.0, 99], labels=["< -1 s", "-1..0 s", "sau khi chạm"])
display(df.pivot_table(index="pha", columns="kết quả", values=["can_lift_mean", "occ_ratio"], aggfunc="mean",
                       observed=False).round(2))

display(Markdown("### Từng frame"))
display(df[["scene", "t_rel"] + cols[1:]].round(2))


def red_mask(img):
    a = np.asarray(img).astype(np.int16)
    return (a[..., 0] > 45) & (a[..., 0] > 2 * a[..., 1]) & (a[..., 0] > 2 * a[..., 2])


def overlay(ax, img, cm, title):
    h, w = img.shape[:2]
    ax.imshow(img)
    ax.imshow(cm / max(float(cm.max()), 1e-9), cmap="jet", alpha=0.45, extent=(0, w, h, 0), interpolation="bilinear")
    m = red_mask(img)
    if m.any():
        ax.contour(m, levels=[0.5], colors="white", linewidths=0.8)
    ax.set_title(title, fontsize=8)
    ax.axis("off")


for scene in df["scene"].unique():
    idx = np.where(df["scene"] == scene)[0]
    fig, axes = plt.subplots(len(idx), 4, figsize=(13, 2.6 * len(idx)), squeeze=False)
    for r, i in enumerate(idx):
        row = df.iloc[i]
        cm = arr["cellmaps"][i].astype(np.float32)
        img = arr["images"][i]
        axes[r, 0].imshow(img)
        axes[r, 0].set_title(f"t = {row['t_rel']:+.1f} s so với lúc chạm | lon {row['can_px']} px", fontsize=8)
        axes[r, 0].axis("off")
        overlay(axes[r, 1], img, cm[0], f"block 0 | can_lift {row['can_lift_b0']:.2f}")
        overlay(axes[r, 2], img, cm.mean(0), f"trung bình các block | can_lift {row['can_lift_mean']:.2f}")
        axes[r, 3].imshow(arr["occluded"][i])
        axes[r, 3].set_title(f"che lon | occ_ratio {row['occ_ratio']:.2f} | nắm {row['closure_orig']:.2f}"
                             f" -> {row['closure_occ']:.2f}" if not np.isnan(row["occ_ratio"])
                             else "lon không thấy trong ảnh: không che", fontsize=8)
        axes[r, 3].axis("off")
    fig.suptitle(scene, fontsize=10)
    fig.tight_layout(rect=(0, 0, 1, 1 - 0.35 / fig.get_figheight()))
    fig.savefig(OUT / f"fig_{scene}.png", dpi=90)
    plt.show()


### Cách đọc kết quả

- **`can_lift` lớn hơn 1 rõ rệt ở cả cảnh thành công lẫn thất bại**: mô hình có chú ý vào lon. Khi đó lỗi đổ lon nằm ở độ chính
  xác của chuyển động tay, không phải ở việc nhìn thấy lon.
- **`can_lift` của cảnh thất bại thấp hơn hẳn cảnh thành công**, nhất là trong khoảng 1 s trước khi chạm: mô hình mất dấu lon đúng
  lúc cần nhắm tay.
- **`occ_ratio` khoảng 1 trong khi `can_lift` cao**: attention có rơi vào lon nhưng action không phụ thuộc vào nó. Tay đi theo
  thói quen học từ dữ liệu (quỹ đạo trung bình), không chỉnh theo vị trí lon.
- **`closure` gần như không đổi khi che lon**: quyết định nắm tay không dựa trên hình ảnh lon (có thể dựa trên state hay thời
  điểm).
- `can_px` rất nhỏ (vài pixel) nghĩa là lon gần như khuất khỏi camera đầu (bị tay che hoặc ở mép ảnh). Các frame đó không đo
  được attention vào lon, và cũng là dấu hiệu đáng chú ý: mô hình phải nhắm tay mà không thấy lon.

Giới hạn: attention chỉ cho biết mô hình *nhìn* vào đâu, không chứng minh nó *dùng* thông tin đó. Phép che lon kiểm tra trực tiếp
hơn. Số frame ít (vài frame mỗi cảnh), nên chỉ so sánh xu hướng, không kết luận từ một frame.


In [ ]:
# @title 10. Tải kết quả lên Drive
import os
import pathlib

OUT = pathlib.Path("/content/attn/out")
EVAL_NAME = CFG["EVAL_NAME"] or f"{CFG['EXP']}_ckpt{os.environ['CKPT_STEP']}_center_colab"
dest = gd.path(f"{CFG['DRIVE_RUNS']}/attn/{EVAL_NAME}", create=True)
for f in sorted(OUT.iterdir()):
    if f.is_file():
        gd.upload(f, dest)   # replaces a file of the same name from an earlier run
print(f"đã tải lên My Drive/{CFG['DRIVE_RUNS']}/attn/{EVAL_NAME}/ : https://drive.google.com/drive/folders/{dest}")
